# 第07回 ソルバ1：Pythonから線形計画を解く

今日の目標は、前回の工場の生産計画を自分のColabで解き、結果を確かめることです。小レポート・提出はありません。

## 1. 自分のノートブックを用意する

1. Googleアカウントでログインし、「ファイル → ドライブにコピーを保存」を選びます。
2. 自分のDriveに保存したコピーで作業します。
3. コード左の▶か Shift+Enter で、上から順に実行します。

ノートブックは説明のセルとコードのセルからなります。コードの実行場所はGoogle側の計算機です。CPUで十分です。最初の接続には少し時間がかかります。

In [ ]:
1 + 1

上の式を `1 + 3` に変更して再実行してください。結果が4になったら、変更と実行ができています。

## 2. 変数・数値・文字列

`=` は代入です。右側の値に、左側の名前を付けます。文字列は半角の引用符で囲み、`print` で値を表示します。

In [ ]:
product = "製品1"
unit_profit = 4
quantity = 3
profit = unit_profit * quantity
print(product, profit, "万円")

`quantity` はPythonで値を保持する名前です。後でPuLPで作る決定変数は「これからソルバが値を決めるもの」で、役割が違います。

`*` は掛け算、`/` は割り算、`**` は累乗です。数式の掛け算記号を省略して `4x1` と書くことはできません。

### やってみよう1

単位利益5万円の製品を2.5単位作るときの利益を計算する。

1. 下のセルの `unit_profit = 4` を `unit_profit = 5` にする。
2. `quantity = 3` を `quantity = 2.5` にする。
3. **同じセル全体を実行し、表示された利益（万円）を確認する。** 後半の計算式とprintはそのままでよい。

In [ ]:
unit_profit = 4  # 5へ変更
quantity = 3     # 2.5へ変更
profit = unit_profit * quantity
print(profit)

### 解答1
`unit_profit = 5`、`quantity = 2.5` とすれば、`5 * 2.5 = 12.5` 万円です。

## 3. リストと添字

複数の値は `[]` の中にカンマで並べます。添字は0から始まります。

In [ ]:
profits = [4, 5]
amounts = [3, 1]
print(profits[0])
print(profits[1])
print(profits[0]*amounts[0] + profits[1]*amounts[1])

表示は4、5、17です。`profits[2]` は存在せず、IndexErrorになります。数学の製品1・2を、Pythonでは添字0・1で扱っています。

## 4. forと字下げ

`for v in values:` で、各要素を順に `v` として使います。繰り返す行は半角スペース4個で字下げします。

In [ ]:
values = [4, 5, 6]
total = 0
for v in values:
    total = total + v
print(total)

`total = 0` は繰り返しの前に置きます。ループ内に置くと、毎回それまでの合計を消してしまいます。最後の表示は15です。

### やってみよう2

`values = [2, 3, 7]` の合計をforで求める。下のコードセルのコメントの下に、次の役割を持つ**4行**を入力する。

1. 合計 `total` を0にする。
2. `for v in values:` で繰り返しを始める。
3. `total` に `v` を加える。
4. `total` を表示する。

追加する4行のうち、3行目だけを半角スペース4個で字下げする。セル全体を実行し、合計を確認する。

In [ ]:
values = [2, 3, 7]
# 下に初期化、for、printを書いて実行してください。

### 解答2

問題は `[2, 3, 7]` の合計。`total` は **0 → 2 → 5 → 12** と変わる。下のセルだけでも確認できる。

In [ ]:
values = [2, 3, 7]
total = 0
for v in values:
    total = total + v
print(total)  # 12

## 5. 実行順と保存

セルの値は、画面の順ではなく実行した順に変わる。途中の値を変更したら、それを使う計算も再実行する。

次のセルを実行すると、**10.0、12.5** の順に表示される。`unit_profit` を5にしても、計算済みの `profit` は自動で変わらない。掛け算をもう一度実行すると12.5になる。

ノートブックの保存と実行中の変数は別。新しい実行環境になったら上から実行し直す。保存表示を確認し、必要なら Ctrl+S ／ Macでは Cmd+S で保存する。

In [ ]:
unit_profit = 4
quantity = 2.5
profit = unit_profit * quantity
unit_profit = 5
print(profit)
profit = unit_profit * quantity
print(profit)

## 6. PuLPとソルバを準備する

PuLPは目的関数・制約をPythonで記述するライブラリ。HiGHSは、そのモデルを解くソルバ（最適解を計算するプログラム）である。

下の2つのコードセルを順に実行する。1つ目がインストール、2つ目が読み込みと確認。PuLP **3.3.0**、ソルバ一覧に **HiGHS** が表示されれば準備できている。授業で同じ動作を再現するため版を固定している。新しい実行環境では再実行する。

In [ ]:
%pip -q install "pulp==3.3.0" "highspy==1.12.0"

In [ ]:
import pulp
print("PuLP", pulp.__version__)
print("使えるソルバ:", pulp.listSolvers(onlyAvailable=True))

## 7. 前回の工場をモデルにする

製品1・2の生産量 $x_1,x_2$ は連続量、利益は万円です。

$$\begin{aligned}\max\quad &4x_1+5x_2\\\text{制約}\quad &x_1+2x_2\le5\\&2x_1+2x_2\le8\\&x_1+x_2\le10\\&x_1,x_2\ge0\end{aligned}$$

原料A・B・Cの在庫は5・8・10 kg。`LpMaximize` は最大化を指定する定数、`lowBound=0` は非負制約を指定する。種類を指定しない `LpVariable` は連続変数である。`x1`・`x2` の値は自分で代入せず、ソルバが決める。

| 数式での役割 | コードでの指定 |
|---|---|
| 最大化 | `pulp.LpMaximize` |
| 生産量が0以上 | `lowBound=0` |
| 利益 | `4*x1 + 5*x2` |
| 原料A・B・Cの在庫 | 制約の右辺 `5`・`8`・`10` |

In [ ]:
model = pulp.LpProblem("Production", pulp.LpMaximize)
x1 = pulp.LpVariable("x1", lowBound=0)
x2 = pulp.LpVariable("x2", lowBound=0)
model += 4*x1 + 5*x2, "profit"
model += x1 + 2*x2 <= 5, "material_A"
model += 2*x1 + 2*x2 <= 8, "material_B"
model += x1 + x2 <= 10, "material_C"
print(model)

`model += 式` で目的関数を、`model += 不等式` で制約を追加します。これはPuLPが用意した書き方です。目的関数は1つだけ指定します。

等式制約は `==`、代入は `=`。不等号は `<=`・`>=` です。ソルバへ渡すために、必ずスラック変数を自分で追加する必要はありません。

## 8. 解く前後の確認

対象は製品1・2の生産計画。利益を最大化し、生産量は0以上とする。

| 条件 | 確認する式 |
|---|---|
| 利益（万円） | $4x_1+5x_2$ |
| 原料A | $x_1+2x_2\leq5$ |
| 原料B | $2x_1+2x_2\leq8$ |
| 原料C | $x_1+x_2\leq10$ |

第7節のモデル作成セルを実行してから、下のセルを実行する。まず状態が `Optimal` か確認する。`if` の下の字下げした行は、その条件が成立するときだけ実行される。`else` はそれ以外の場合。

In [ ]:
model.solve(pulp.HiGHS(msg=False))
status = pulp.LpStatus[model.status]
print("状態:", status)
if status == "Optimal":
    print("製品1:", pulp.value(x1))
    print("製品2:", pulp.value(x2))
    print("利益:", pulp.value(model.objective), "万円")
else:
    print("最適解は未確認。状態と条件を確認してください。")

得られる**最適解は生産量の組 $(x_1,x_2)=(3,1)$**、**最適値は利益17万円**である。`Optimal` は最適解が得られた状態を表す。

検算では各原料の使用量を在庫と比べ、余りも確認する。

| 原料 | 使用量の計算 | 在庫（kg） | 余り（kg） |
|---|---|---:|---:|
| A | $3+2\times1=5$ | 5 | 0 |
| B | $2\times3+2\times1=8$ | 8 | 0 |
| C | $3+1=4$ | 10 | 6 |

利益は $4\times3+5\times1=17$ 万円。生産量も両方0以上で、すべての条件を満たす。

In [ ]:
if status == "Optimal":
    a, b = pulp.value(x1), pulp.value(x2)
    print("生産量（0以上）:", a, b)
    print("原料A 使用量:", a + 2*b, "在庫: 5", "余り:", 5-(a+2*b))
    print("原料B 使用量:", 2*a + 2*b, "在庫: 8", "余り:", 8-(2*a+2*b))
    print("原料C 使用量:", a + b, "在庫: 10", "余り:", 10-(a+b))
    print("利益:", 4*a + 5*b, "万円")
else:
    print("検算の前に、解いたときの状態を確認してください。")

### やってみよう3

原料Aの在庫だけを **5 kgから6 kg** に変更する。

1. 下のセルの `trial += u1 + 2*u2 <= 5, "material_A"` という行を探す。
2. **この行の右辺5だけを6へ変更**する。
3. 問題の作成から表示まで含む、**このコードセル全体を実行**する。
4. 状態が `Optimal` か、製品1・2の生産量と利益はいくつかを確認する。

`trial` は変更後の問題、`u1`・`u2` はその製品1・2の生産量である。

| 変更後の条件 | 内容 |
|---|---|
| 利益（万円） | $4u_1+5u_2$ を最大化 |
| 原料A | $u_1+2u_2\leq6$ |
| 原料B（変更なし） | $2u_1+2u_2\leq8$ |
| 原料C（変更なし） | $u_1+u_2\leq10$ |
| 生産量（変更なし） | $u_1,u_2\geq0$、連続量 |

In [ ]:
trial = pulp.LpProblem("Production", pulp.LpMaximize)
u1 = pulp.LpVariable("u1", lowBound=0)
u2 = pulp.LpVariable("u2", lowBound=0)
trial += 4*u1 + 5*u2, "profit"
trial += u1 + 2*u2 <= 5, "material_A"
trial += 2*u1 + 2*u2 <= 8, "material_B"
trial += u1 + u2 <= 10, "material_C"
trial.solve(pulp.HiGHS(msg=False))
trial_status = pulp.LpStatus[trial.status]
print("状態:", trial_status)
if trial_status == "Optimal":
    print("製品1:", pulp.value(u1))
    print("製品2:", pulp.value(u2))
    print("利益:", pulp.value(trial.objective), "万円")
else:
    print("最適解は未確認。状態と条件を確認してください。")

### 解答3

原料Aの制約を $u_1+2u_2\leq6$ にすると、最適解は **製品1が2、製品2が2**、利益の最適値は **18万円** になる。

| 原料 | 使用量の計算 | 変更後の在庫（kg） | 余り（kg） |
|---|---|---:|---:|
| A | $2+2\times2=6$ | 6 | 0 |
| B | $2\times2+2\times2=8$ | 8 | 0 |
| C | $2+2=4$ | 10 | 6 |

利益は $4\times2+5\times2=18$ 万円で、変更前の17万円より1万円増える。今回の5→6 kgの範囲では、Aを1 kg増やすと利益が1万円増え、第6回の限界価値と一致する。どこまで増やしても同じ割合とは限らない。

表示が3・1・17のままなら、原料Aの右辺とセル全体を再実行したかを確認する。

## 9. よくあるエラーを直す

| 表示・症状 | 確認すること |
|---|---|
| NameError | 名前の綴り、値を定義したセルを実行したか |
| ModuleNotFoundError | インストール用セルとimportを実行したか |
| IndentationError | for・ifの字下げが揃っているか |
| 結果が変わらない | 編集したセルと、その後の計算を再実行したか |

エラー表示の最後の行と、示されたコードの行から確認します。

| ソルバの状態 | 意味 | 確認すること |
|---|---|---|
| Infeasible | 条件を満たす解がない | 在庫・係数・不等号 |
| Unbounded | 目的値を際限なく改善できる | 制約の入れ忘れ・最大化と最小化 |
| Not Solved / Undefined | 最適解が得られたと確認できない | solveの実行、直前のメッセージ |

`Optimal` 以外で表示された数値を、そのまま最適解とみなさない。

## まとめと次回

数式 → PuLPのモデル → ソルバ → 状態の確認 → 元の条件で検算、の順です。保存したノートブックを開き直せることを確認してください。次回も端末を使い、表のデータを扱って輸送問題を解きます。

参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html)、[PuLP](https://github.com/coin-or/pulp/tree/3.3.0)、[HiGHS](https://highs.dev/)。